In [1]:
from envinit import Workspace, Configuration
config = Configuration()
workspace = Workspace()
workspace.init()

INFO	Workspace initialized successfully.


In [2]:
from transformers import AutoProcessor, Gemma3ForConditionalGeneration, pipeline
from transformers import AutoTokenizer, AutoModelForCausalLM, Trainer, TrainingArguments, DataCollatorForLanguageModeling
from datasets import Dataset, DatasetDict
from PIL import Image
import requests
import torch
import pandas as pd
from collections import Counter
import ast
from sklearn.model_selection import train_test_split

In [3]:
df = pd.read_excel("annotated_full.xlsx")
df = df.dropna(subset=["Content"])

In [4]:
def prepare_inputs(df):

    inputs = []
    for row in df.iterrows():
        prompt = row[1]["Prompt"]
        style = ", ".join(ast.literal_eval(row[1]["Style"]))
        persons = ", ".join(ast.literal_eval(row[1]["PER"]))
        organizations = ", ".join(ast.literal_eval(row[1]["ORG"]))
        objects = ", ".join(ast.literal_eval(row[1]["OBJ"]))
        concepts = ", ".join(ast.literal_eval(row[1]["MISC"]))
        title = row[1]["Title"]
        body = row[1]["Content"]

        template = f"""<bos><start_of_turn>user
Sen bir Zaytung haber yazarısın. Aşağıda verilen konu ve bilgiler doğrultusunda yeni bir haberi mizahi bir şekilde oluşturmalısın.

{prompt}

Üslüp: {style}
Kişiler: {persons}
Kurumlar: {organizations}
Objeler: {objects}
Konseptler: {concepts}<end_of_turn>
<start_of_turn>model
Başlık: {title}

Haber: {body}<end_of_turn><eos>"""

        inputs.append(template)

    return inputs

In [5]:
inputs = prepare_inputs(df)

In [6]:
train_val, test = train_test_split(inputs, test_size=0.2, random_state=42)
train, val = train_test_split(train_val, test_size=0.1, random_state=42)

In [ ]:
import random
inputs = [input.split("Başlık")[0][:-1] for input in random.sample(test, 100)]

In [16]:
model_name = "/home/ubuntu/shared/hf_home/hub/gemma-3-4b-ft-2/checkpoint-1100"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = Gemma3ForConditionalGeneration.from_pretrained(model_name, torch_dtype=torch.bfloat16, attn_implementation="eager").eval().to("cuda")

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

In [ ]:
tokenized_input = tokenizer(inputs, return_tensors="pt", padding=True, truncation=True).to("cuda")

In [ ]:
with torch.inference_mode():
    generations = model.generate(**tokenized_input, max_new_tokens=2048, temperature=0.7, top_p=0.90, top_k=64)

In [54]:
outputs = tokenizer.batch_decode(generation, skip_special_tokens=True)

In [ ]:
with open("test_output.txt", "w", encoding="utf-8") as file:
    for item in outputs:
        file.write(item + "\n"+"~-"*20+"\n")